# Lending Club Interactive Loan Default Predictor

This notebook loads trained artifacts and provides an interactive dashboard where a user can enter applicant/loan information and get:
 
1. Predicted probability of default
2. Threshold-based default flag
3. Visual risk gauge# Lending Club Interactive Loan Default Predictor

This notebook loads trained artifacts and lets users input applicant/loan details to predict default risk.

Outputs:
1. Predicted default probability
2. Threshold-based prediction (likely default vs non-default)
3. Risk band + gauge chart

Required:
- `artifacts/models/trained_models_bundle.joblib`
 
Required files:
- `artifacts/cleaning_only/cleaning_pipelines.joblib`
- `artifacts/models/trained_models_bundle.joblib`

In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Optional wider notebook area
display(HTML("<style>.container { width: 95% !important; }</style>"))

In [2]:
# Load model artifacts 
MODEL_DIR = Path("artifacts/models")
TRAINED_BUNDLE_PATH = MODEL_DIR / "trained_models_bundle.joblib"

if not TRAINED_BUNDLE_PATH.exists():
    raise FileNotFoundError(f"Missing {TRAINED_BUNDLE_PATH}. Run updated data_modeling.ipynb first.")

trained_bundle = joblib.load(TRAINED_BUNDLE_PATH)

print("Loaded trained bundle.")
print("Bundle keys:", list(trained_bundle.keys()))
print("Threshold keys:", list(trained_bundle.get("thresholds", {}).keys()))

Loaded trained bundle.
Bundle keys: ['meta', 'summary', 'log_fund', 'xgb_fund', 'xgb_full', 'enc_fund_log', 'enc_fund_xgb', 'enc_full_xgb', 'thresholds']
Threshold keys: ['log_fund_val_best_f1_thr', 'xgb_fund_val_best_f1_thr', 'xgb_full_val_best_f1_thr']


In [3]:
# Model specs + helper functions + prediction core
MODEL_SPECS = {
    "Logistic (Fundamental)": {
        "model_key": "log_fund",
        "encoder_key": "enc_fund_log",
        "threshold_key": "log_fund_val_best_f1_thr",
        "uses_full_fields": False,
    },
    "XGB (Fundamental)": {
        "model_key": "xgb_fund",
        "encoder_key": "enc_fund_xgb",
        "threshold_key": "xgb_fund_val_best_f1_thr",
        "uses_full_fields": False,
    },
    "XGB (Full)": {
        "model_key": "xgb_full",
        "encoder_key": "enc_full_xgb",
        "threshold_key": "xgb_full_val_best_f1_thr",
        "uses_full_fields": True,
    },
}

for name, spec in MODEL_SPECS.items():
    if spec["model_key"] not in trained_bundle:
        raise KeyError(f"{name}: missing model key '{spec['model_key']}'")
    if spec["encoder_key"] not in trained_bundle:
        raise KeyError(f"{name}: missing encoder key '{spec['encoder_key']}'")

thresholds = trained_bundle.get("thresholds", {})


def _safe_float(x):
    try:
        if x is None or (isinstance(x, str) and x.strip() == ""):
            return np.nan
        return float(x)
    except Exception:
        return np.nan


def _norm_cat(x):
    if x is None:
        return np.nan
    s = str(x).strip()
    return np.nan if s == "" else s.lower()


def _term_to_num(term):
    if term is None:
        return np.nan
    m = re.search(r"(\d+)", str(term))
    return float(m.group(1)) if m else np.nan


def _emp_to_years(emp):
    if emp is None:
        return np.nan
    s = str(emp).strip().lower()
    if s == "":
        return np.nan
    s = re.sub(r"10\+", "10", s)
    s = re.sub(r"<\s*1", "0", s)
    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


def _parse_mon_yyyy(s, field_name):
    if s is None or str(s).strip() == "":
        return pd.NaT
    try:
        return pd.to_datetime(str(s).strip(), format="%b-%Y")
    except Exception:
        raise ValueError(f"{field_name} must be Mon-YYYY (example: Jan-2016).")


def _risk_band(p):
    if p < 0.05:
        return "Low"
    elif p < 0.15:
        return "Moderate"
    elif p < 0.30:
        return "Elevated"
    return "High"


def _fmt_metric(x):
    return f"{float(x):.3f}"


def _get_human_loop_metrics(model_name):
    summary = trained_bundle.get("summary", None)
    if summary is None or len(summary) == 0:
        raise ValueError("Missing summary in trained bundle. Re-run updated data_modeling.ipynb.")

    required_cols = [
        "model",
        "val_best_f1",
        "val_accuracy_at_best_f1_thr",
        "val_recall_at_best_f1_thr",
        "val_precision_at_best_f1_thr",
        "val_best_thr_f1",
    ]
    missing = [c for c in required_cols if c not in summary.columns]
    if missing:
        raise ValueError(
            f"Summary missing required columns: {missing}. "
            "Please re-run updated data_modeling.ipynb."
        )

    row = summary.loc[summary["model"] == model_name]
    if row.empty:
        raise ValueError(f"No summary row found for model '{model_name}'")
    row = row.iloc[0]

    return {
        "f1": float(row["val_best_f1"]),
        "accuracy": float(row["val_accuracy_at_best_f1_thr"]),
        "recall": float(row["val_recall_at_best_f1_thr"]),
        "precision": float(row["val_precision_at_best_f1_thr"]),
        "threshold": float(row["val_best_thr_f1"]),
    }


def _get_required_cols_from_encoder(encoder):
    if hasattr(encoder, "feature_names_in_"):
        return list(encoder.feature_names_in_)

    cols = []
    for name, trans, c in encoder.transformers_:
        if name == "remainder":
            continue
        if isinstance(c, (list, tuple, np.ndarray, pd.Index)):
            cols.extend(list(c))

    out, seen = [], set()
    for c in cols:
        if c not in seen:
            out.append(c)
            seen.add(c)

    if not out:
        raise ValueError("Could not infer expected input columns from encoder.")
    return out


def _get_num_cols_from_encoder(encoder):
    for name, trans, cols in encoder.transformers_:
        if name == "num":
            return list(cols)
    return []


def _build_input_row_for_encoder(encoder, form_data):
    req_cols = _get_required_cols_from_encoder(encoder)
    num_cols = _get_num_cols_from_encoder(encoder)

    row = {c: np.nan for c in req_cols}

    def put(k, v):
        if k in row:
            row[k] = v

    annual_inc = _safe_float(form_data.get("annual_inc"))
    open_acc = _safe_float(form_data.get("open_acc"))
    total_acc = _safe_float(form_data.get("total_acc"))
    delinq_2yrs = _safe_float(form_data.get("delinq_2yrs"))
    pub_rec = _safe_float(form_data.get("pub_rec"))
    dti = _safe_float(form_data.get("dti"))
    revol_bal = _safe_float(form_data.get("revol_bal"))
    revol_util = _safe_float(form_data.get("revol_util"))
    loan_amnt = _safe_float(form_data.get("loan_amnt"))
    int_rate = _safe_float(form_data.get("int_rate"))
    installment = _safe_float(form_data.get("installment"))
    term = _term_to_num(form_data.get("term"))
    emp_length_years = _emp_to_years(form_data.get("emp_length"))

    fico_low = _safe_float(form_data.get("fico_range_low"))
    fico_high = _safe_float(form_data.get("fico_range_high"))
    fico = np.nan if np.isnan(fico_low) or np.isnan(fico_high) else (fico_low + fico_high) / 2.0

    issue_dt = _parse_mon_yyyy(form_data.get("issue_d"), "Issue date")
    earliest_dt = _parse_mon_yyyy(form_data.get("earliest_cr_line"), "Earliest credit line")

    if pd.notna(issue_dt):
        issue_month = float(issue_dt.month)
        issue_weekday = float(issue_dt.weekday())
        issue_quarter = float(issue_dt.quarter)
        issue_weekofyear = float(issue_dt.isocalendar().week)
        issue_year = float(issue_dt.year)
    else:
        issue_month = issue_weekday = issue_quarter = issue_weekofyear = issue_year = np.nan

    if pd.notna(issue_dt) and pd.notna(earliest_dt):
        credit_age_years = (issue_dt - earliest_dt).days / 365.25
        if credit_age_years < 0 or credit_age_years > 100:
            credit_age_years = np.nan
    else:
        credit_age_years = np.nan

    if pd.notna(credit_age_years):
        credit_age_bucket = pd.cut(
            pd.Series([credit_age_years]),
            bins=[-1, 1, 3, 5, 10, 20, 100],
            labels=False
        ).iloc[0]
        credit_age_bucket = float(credit_age_bucket) if pd.notna(credit_age_bucket) else np.nan
    else:
        credit_age_bucket = np.nan

    revol_to_loan = np.nan
    if pd.notna(revol_bal) and pd.notna(loan_amnt) and loan_amnt != 0:
        revol_to_loan = revol_bal / loan_amnt

    installment_to_monthly_income = np.nan
    if pd.notna(installment) and pd.notna(annual_inc) and annual_inc != 0:
        installment_to_monthly_income = installment / (annual_inc / 12.0)

    open_to_total_ratio = np.nan
    if pd.notna(open_acc) and pd.notna(total_acc) and total_acc != 0:
        open_to_total_ratio = open_acc / total_acc

    fico_times_revolutil = np.nan
    if pd.notna(fico) and pd.notna(revol_util):
        fico_times_revolutil = fico * (revol_util / 100.0)

    revol_util_gt_100 = np.nan
    if pd.notna(revol_util):
        revol_util_gt_100 = float(revol_util > 100)

    # Numeric features
    put("annual_inc", annual_inc)
    put("open_acc", open_acc)
    put("total_acc", total_acc)
    put("delinq_2yrs", delinq_2yrs)
    put("pub_rec", pub_rec)
    put("dti", dti)
    put("revol_bal", revol_bal)
    put("revol_util", revol_util)
    put("loan_amnt", loan_amnt)
    put("term", term)
    put("int_rate", int_rate)
    put("installment", installment)
    put("fico", fico)
    put("credit_age_years", credit_age_years)
    put("emp_length_years", emp_length_years)

    put("issue_d_month", issue_month)
    put("issue_d_weekday", issue_weekday)
    put("issue_d_quarter", issue_quarter)
    put("issue_d_weekofyear", issue_weekofyear)
    put("issue_d_year", issue_year)

    put("revol_to_loan", revol_to_loan)
    put("installment_to_monthly_income", installment_to_monthly_income)
    put("open_to_total_ratio", open_to_total_ratio)
    put("fico_times_revolutil", fico_times_revolutil)
    put("revol_util_gt_100", revol_util_gt_100)
    put("credit_age_bucket", credit_age_bucket)

    # Categorical features
    put("home_ownership", _norm_cat(form_data.get("home_ownership")))
    put("verification_status", _norm_cat(form_data.get("verification_status")))
    put("purpose", _norm_cat(form_data.get("purpose")))
    put("addr_state", _norm_cat(form_data.get("addr_state")))
    put("grade", _norm_cat(form_data.get("grade")))
    put("sub_grade", _norm_cat(form_data.get("sub_grade")))

    # Missing-indicator columns if present
    for c in req_cols:
        if c.endswith("_missing"):
            base = c[:-8]
            row[c] = 1.0 if pd.isna(row.get(base, np.nan)) else 0.0

    X = pd.DataFrame([row], columns=req_cols)
    if num_cols:
        X[num_cols] = X[num_cols].apply(pd.to_numeric, errors="coerce")

    return X


def _apply_model_specific_inference_transforms(model_name, X_row):
    """
    Aligns inference input with training transformations.
    """
    X = X_row.copy()

    if model_name == "Logistic (Fundamental)":
        # Match logistic wrangling log transform
        for c in ["annual_inc", "revol_bal", "loan_amnt"]:
            if c in X.columns:
                vals = pd.to_numeric(X[c], errors="coerce")
                X[c] = np.log1p(vals.clip(lower=0))

    return X


def predict_from_user_input(model_name, form_data):
    spec = MODEL_SPECS[model_name]
    model = trained_bundle[spec["model_key"]]
    encoder = trained_bundle[spec["encoder_key"]]

    # Use threshold from summary if available; fallback to thresholds dict
    h = _get_human_loop_metrics(model_name)
    threshold = float(h["threshold"]) if "threshold" in h else float(thresholds.get(spec["threshold_key"], 0.50))

    X_row = _build_input_row_for_encoder(encoder, form_data)
    X_row = _apply_model_specific_inference_transforms(model_name, X_row)
    X_enc = encoder.transform(X_row)

    p_default = float(model.predict_proba(X_enc)[:, 1][0])
    pred_class = int(p_default >= threshold)

    return {
        "prob_default": p_default,
        "threshold": threshold,
        "pred_class": pred_class,
        "risk_band": _risk_band(p_default),
    }

In [4]:
# model summary
summary = trained_bundle.get("summary", None)
if summary is not None:
    display(summary.sort_values("val_pr_auc", ascending=False).reset_index(drop=True))
else:
    print("No summary table found in trained bundle.")

,model,val_roc_auc,val_pr_auc,val_brier,test_roc_auc,test_pr_auc,test_brier,val_best_thr_f1,val_best_f1,val_accuracy_at_best_f1_thr,val_precision_at_best_f1_thr,val_recall_at_best_f1_thr,test_accuracy_at_val_best_f1_thr,test_precision_at_val_best_f1_thr,test_recall_at_val_best_f1_thr,test_f1_at_val_best_f1_thr
0,XGB (Full),0.717307,0.156310,0.222553,0.712738,0.141573,0.218918,0.674875,0.231128,0.829579,0.166484,0.377834,0.840297,0.152520,0.350048,0.212466
1,XGB (Fundamental),0.672315,0.126593,0.219220,0.666881,0.110836,0.210933,0.598504,0.195307,0.790295,0.131990,0.375387,0.804704,0.118303,0.336806,0.175102
2,Logistic (Fundamental),0.635444,0.107120,0.315952,0.634862,0.096417,0.314962,0.670678,0.173418,0.725587,0.108959,0.424613,0.727770,0.097923,0.416881,0.158594


In [5]:
# Validation rules (including model-specific checks)
# Optional clamp mode:
# False -> raise error if out of range
# True  -> clamp out-of-range values and continue with warning
CLAMP_OUT_OF_RANGE = False  # True => auto-clamp + warnings; False => strict errors

BASE_RANGES = {
    "fico_range_low": (300, 900),
    "fico_range_high": (300, 900),
    "loan_amnt": (500, 40000),
    "annual_inc": (1000, 5_000_000),
    "open_acc": (0, 100),
    "total_acc": (0, 250),
    "delinq_2yrs": (0, 50),
    "pub_rec": (0, 50),
    "dti": (0, 100),
    "revol_bal": (0, 2_000_000),
    "revol_util": (0, 200),
}

FULL_MODEL_RANGES = {
    "int_rate": (0, 40),
    "installment": (1, 5000),
}


def _validate_range(field_name, value, lo, hi, clamp=False, warnings=None):
    if warnings is None:
        warnings = []
    v = _safe_float(value)
    if pd.isna(v):
        raise ValueError(f"{field_name} is required.")
    if v < lo or v > hi:
        if clamp:
            old_v = v
            v = float(np.clip(v, lo, hi))
            warnings.append(f"{field_name}={old_v} was outside [{lo}, {hi}] and clamped to {v}.")
        else:
            raise ValueError(f"{field_name} must be between {lo} and {hi}.")
    return v


def validate_form_by_model(model_name, form_data, clamp=False):
    f = dict(form_data)
    warnings = []

    # Base checks
    for field, (lo, hi) in BASE_RANGES.items():
        f[field] = _validate_range(field, f.get(field), lo, hi, clamp=clamp, warnings=warnings)

    # Explicit logistic checks requested
    if model_name == "Logistic (Fundamental)":
        if not (300 <= f["fico_range_low"] <= 900):
            raise ValueError("FICO low must be between 300 and 900")
        if not (300 <= f["fico_range_high"] <= 900):
            raise ValueError("FICO high must be between 300 and 900")

    # Logical checks
    if f["fico_range_high"] < f["fico_range_low"]:
        if clamp:
            a, b = f["fico_range_low"], f["fico_range_high"]
            f["fico_range_low"], f["fico_range_high"] = b, a
            warnings.append("fico_range_high < fico_range_low; values swapped.")
        else:
            raise ValueError("fico_range_high must be >= fico_range_low.")

    if f["open_acc"] > f["total_acc"]:
        if clamp:
            f["total_acc"] = f["open_acc"]
            warnings.append("open_acc > total_acc; total_acc set to open_acc.")
        else:
            raise ValueError("open_acc cannot exceed total_acc.")

    issue_dt = _parse_mon_yyyy(f.get("issue_d"), "Issue date")
    earliest_dt = _parse_mon_yyyy(f.get("earliest_cr_line"), "Earliest credit line")
    if pd.notna(issue_dt) and pd.notna(earliest_dt) and earliest_dt > issue_dt:
        raise ValueError("earliest_cr_line must be <= issue_d.")

    # Full-model checks
    if model_name == "XGB (Full)":
        for field, (lo, hi) in FULL_MODEL_RANGES.items():
            f[field] = _validate_range(field, f.get(field), lo, hi, clamp=clamp, warnings=warnings)

        grade = str(f.get("grade", "")).upper().strip()
        sub_grade = str(f.get("sub_grade", "")).upper().strip()

        if grade not in list("ABCDEFG"):
            raise ValueError("grade must be one of A-G.")
        if not re.fullmatch(r"[A-G][1-5]", sub_grade):
            raise ValueError("sub_grade must be in format A1..G5.")
        if sub_grade[0] != grade:
            raise ValueError("sub_grade must match grade initial (e.g., C -> C1..C5).")

    return f, warnings

In [ ]:
# Widgets (full labels + wide sizing)
INPUT_LAYOUT = widgets.Layout(width="150px", height="30px")

def make_row(label_text, widget):
    label = widgets.HTML(f"<div style='min-width:260px; font-size:13px;'><b>{label_text}</b></div>")
    return widgets.HBox([label, widget], layout=widgets.Layout(width="100%", align_items="center"))

model_w = widgets.Dropdown(
    options=list(MODEL_SPECS.keys()),
    value="XGB (Fundamental)",
    description="",
    layout=widgets.Layout(width="170px", height="30px")
)

loan_amnt_w = widgets.FloatText(value=12000, description="", layout=INPUT_LAYOUT)
term_w = widgets.Dropdown(options=["36 months", "60 months"], value="36 months", description="", layout=INPUT_LAYOUT)
annual_inc_w = widgets.FloatText(value=75000, description="", layout=INPUT_LAYOUT)
emp_length_w = widgets.Dropdown(
    options=["< 1 year","1 year","2 years","3 years","4 years","5 years","6 years","7 years","8 years","9 years","10+ years"],
    value="5 years", description="", layout=INPUT_LAYOUT
)
home_ownership_w = widgets.Dropdown(
    options=["RENT", "OWN", "MORTGAGE", "OTHER", "ANY", "NONE"],
    value="MORTGAGE", description="", layout=INPUT_LAYOUT
)
verification_status_w = widgets.Dropdown(
    options=["Not Verified", "Source Verified", "Verified"],
    value="Verified", description="", layout=INPUT_LAYOUT
)
purpose_w = widgets.Dropdown(
    options=[
        "debt_consolidation","credit_card","home_improvement","major_purchase","small_business",
        "car","medical","moving","vacation","house","wedding","renewable_energy","educational","other"
    ],
    value="debt_consolidation", description="", layout=INPUT_LAYOUT
)
addr_state_w = widgets.Text(value="CA", description="", layout=INPUT_LAYOUT)

fico_low_w = widgets.IntText(value=680, description="", layout=INPUT_LAYOUT)
fico_high_w = widgets.IntText(value=684, description="", layout=INPUT_LAYOUT)
earliest_cr_line_w = widgets.Text(value="Jan-2010", description="", layout=INPUT_LAYOUT)
issue_d_w = widgets.Text(value="Jan-2016", description="", layout=INPUT_LAYOUT)

open_acc_w = widgets.IntText(value=10, description="", layout=INPUT_LAYOUT)
total_acc_w = widgets.IntText(value=24, description="", layout=INPUT_LAYOUT)
delinq_2yrs_w = widgets.IntText(value=0, description="", layout=INPUT_LAYOUT)
pub_rec_w = widgets.IntText(value=0, description="", layout=INPUT_LAYOUT)
dti_w = widgets.FloatText(value=18.0, description="", layout=INPUT_LAYOUT)
revol_bal_w = widgets.FloatText(value=12000, description="", layout=INPUT_LAYOUT)
revol_util_w = widgets.FloatText(value=45.0, description="", layout=INPUT_LAYOUT)

# Intentionally disabled unless XGB (Full)
int_rate_w = widgets.FloatText(value=5.5, description="", layout=INPUT_LAYOUT)
installment_w = widgets.FloatText(value=350.0, description="", layout=INPUT_LAYOUT)
grade_w = widgets.Dropdown(options=list("ABCDEFG"), value="A", description="", layout=INPUT_LAYOUT)
sub_grade_w = widgets.Dropdown(
    options=[f"{g}{i}" for g in "ABCDEFG" for i in range(1, 6)],
    value="A1", description="", layout=INPUT_LAYOUT
)

predict_btn = widgets.Button(description="Predict", button_style="danger", icon="calculator", layout=widgets.Layout(width="110px", height="32px"))
reset_btn = widgets.Button(description="Reset", icon="refresh", layout=widgets.Layout(width="110px", height="32px"))

model_hint = widgets.HTML()
out = widgets.Output()
full_only_widgets = [int_rate_w, installment_w, grade_w, sub_grade_w]

In [7]:
# Callbacks + display dashboard
def _update_full_fields_enabled(*args):
    use_full = MODEL_SPECS[model_w.value]["uses_full_fields"]
    for w in full_only_widgets:
        w.disabled = not use_full

    if use_full:
        model_hint.value = "<span style='color:#0b5394; font-size:12px;'>Full-model fields are enabled.</span>"
    else:
        model_hint.value = "<span style='color:#666; font-size:12px;'>Full-model fields are disabled for this model.</span>"

model_w.observe(_update_full_fields_enabled, names="value")
_update_full_fields_enabled()


def _collect_form_data():
    return {
        "loan_amnt": loan_amnt_w.value,
        "term": term_w.value,
        "annual_inc": annual_inc_w.value,
        "emp_length": emp_length_w.value,
        "home_ownership": home_ownership_w.value,
        "verification_status": verification_status_w.value,
        "purpose": purpose_w.value,
        "addr_state": addr_state_w.value.strip().upper() if isinstance(addr_state_w.value, str) else addr_state_w.value,

        "fico_range_low": fico_low_w.value,
        "fico_range_high": fico_high_w.value,
        "earliest_cr_line": earliest_cr_line_w.value.strip() if isinstance(earliest_cr_line_w.value, str) else earliest_cr_line_w.value,
        "issue_d": issue_d_w.value.strip() if isinstance(issue_d_w.value, str) else issue_d_w.value,

        "open_acc": open_acc_w.value,
        "total_acc": total_acc_w.value,
        "delinq_2yrs": delinq_2yrs_w.value,
        "pub_rec": pub_rec_w.value,
        "dti": dti_w.value,
        "revol_bal": revol_bal_w.value,
        "revol_util": revol_util_w.value,

        "int_rate": int_rate_w.value,
        "installment": installment_w.value,
        "grade": grade_w.value,
        "sub_grade": sub_grade_w.value,
    }


def _render_gauge(p, thr):
    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=p,
        number={"valueformat": ".2%"},
        title={"text": "Predicted Default Probability"},
        gauge={
            "axis": {"range": [0, 1]},
            "bar": {"color": "crimson"},
            "steps": [
                {"range": [0.00, 0.05], "color": "#E8F5E9"},
                {"range": [0.05, 0.15], "color": "#FFF9C4"},
                {"range": [0.15, 0.30], "color": "#FFE0B2"},
                {"range": [0.30, 1.00], "color": "#FFCDD2"},
            ],
            "threshold": {"line": {"color": "black", "width": 4}, "thickness": 0.8, "value": thr},
        },
    ))
    fig.update_layout(template="plotly_white", height=320, margin=dict(l=20, r=20, t=40, b=20))
    return fig


def on_predict(_):
    with out:
        clear_output(wait=True)
        try:
            form = _collect_form_data()

            form_valid, warns = validate_form_by_model(
                model_name=model_w.value,
                form_data=form,
                clamp=CLAMP_OUT_OF_RANGE
            )

            print("Warning: Inputs outside realistic range may affect prediction.")
            if len(warns) > 0:
                print("Auto-adjustments:")
                for w in warns:
                    print(f"- {w}")

            result = predict_from_user_input(model_w.value, form_valid)
            p = result["prob_default"]
            thr = result["threshold"]
            pred = result["pred_class"]
            band = result["risk_band"]

            verdict = "LIKELY DEFAULT" if pred == 1 else "LIKELY NON-DEFAULT"

            print(f"Model: {model_w.value}")
            print(f"Predicted default probability: {p:.4f} ({p:.2%})")
            print(f"Decision threshold: {thr:.4f}")
            print(f"Prediction: {verdict}")
            print(f"Risk band: {band}")
            print("Note: Prediction is based on historical patterns and should be used as a risk indicator, not a final decision.")

            # Human-in-the-loop metrics (no N/A)
            m = _get_human_loop_metrics(model_w.value)
            print("\nHuman-in-the-loop disclaimer:")
            print(
                f"- Validation metrics at operating threshold: "
                f"F1={_fmt_metric(m['f1'])}, "
                f"Accuracy={_fmt_metric(m['accuracy'])}, "
                f"Recall={_fmt_metric(m['recall'])}, "
                f"Precision={_fmt_metric(m['precision'])}"
            )
            print("- This prediction is advisory only and must not be used as the sole basis for approval/denial.")
            print("- Review borrower context, policy constraints, and fairness/compliance requirements before final decision.")
            print("- Accuracy can be misleading under class imbalance; use recall/precision trade-offs aligned with policy.")

            display(_render_gauge(p, thr))

        except Exception as e:
            print("Prediction error:", str(e))
            print("Tips:")
            print("- Use Mon-YYYY format (e.g., Jan-2016)")
            print("- Keep numeric inputs in realistic ranges")
            print("- Re-run updated data_modeling.ipynb if summary columns are missing")


def on_reset(_):
    model_w.value = "XGB (Fundamental)"

    loan_amnt_w.value = 12000
    term_w.value = "36 months"
    annual_inc_w.value = 75000
    emp_length_w.value = "5 years"
    home_ownership_w.value = "MORTGAGE"
    verification_status_w.value = "Verified"
    purpose_w.value = "debt_consolidation"
    addr_state_w.value = "CA"

    fico_low_w.value = 680
    fico_high_w.value = 684
    earliest_cr_line_w.value = "Jan-2010"
    issue_d_w.value = "Jan-2016"

    open_acc_w.value = 10
    total_acc_w.value = 24
    delinq_2yrs_w.value = 0
    pub_rec_w.value = 0
    dti_w.value = 18.0
    revol_bal_w.value = 12000
    revol_util_w.value = 45.0

    int_rate_w.value = 5.5
    installment_w.value = 350.0
    grade_w.value = "A"
    sub_grade_w.value = "A1"

    with out:
        clear_output(wait=True)
        print("Form reset complete.")

predict_btn.on_click(on_predict)
reset_btn.on_click(on_reset)

left = widgets.VBox(
    [
        make_row("Model", model_w),
        make_row("Loan amount [loan_amnt]", loan_amnt_w),
        make_row("Loan term [term]", term_w),
        make_row("Annual income [annual_inc]", annual_inc_w),
        make_row("Employment length [emp_length]", emp_length_w),
        make_row("Home ownership [home_ownership]", home_ownership_w),
        make_row("Income verification status [verification_status]", verification_status_w),
        make_row("Loan purpose [purpose]", purpose_w),
        make_row("Borrower state [addr_state]", addr_state_w),
    ],
    layout=widgets.Layout(width="33%", min_width="420px")
)

middle = widgets.VBox(
    [
        make_row("FICO range low [fico_range_low]", fico_low_w),
        make_row("FICO range high [fico_range_high]", fico_high_w),
        make_row("Earliest credit line [earliest_cr_line]", earliest_cr_line_w),
        make_row("Issue date [issue_d]", issue_d_w),
        make_row("Open accounts [open_acc]", open_acc_w),
        make_row("Total accounts [total_acc]", total_acc_w),
        make_row("Delinquencies in 2 years [delinq_2yrs]", delinq_2yrs_w),
        make_row("Public records [pub_rec]", pub_rec_w),
        make_row("Debt-to-income ratio [dti]", dti_w),
        make_row("Revolving balance [revol_bal]", revol_bal_w),
        make_row("Revolving utilization percent [revol_util]", revol_util_w),
    ],
    layout=widgets.Layout(width="33%", min_width="420px")
)

right = widgets.VBox(
    [
        widgets.HTML("<b>Used only by XGB (Full)</b>"),
        model_hint,
        make_row("Interest rate percent [int_rate]", int_rate_w),
        make_row("Installment amount [installment]", installment_w),
        make_row("Loan grade [grade]", grade_w),
        make_row("Loan sub-grade [sub_grade]", sub_grade_w),
        widgets.HBox([predict_btn, reset_btn], layout=widgets.Layout(width="240px", justify_content="space-between")),
    ],
    layout=widgets.Layout(width="33%", min_width="420px")
)

dashboard = widgets.HBox(
    [left, middle, right],
    layout=widgets.Layout(width="100%", flex_flow="row wrap", gap="14px", align_items="flex-start")
)

display(dashboard)
display(out)

Output()

### Notes
- Date fields must be `Mon-YYYY` (e.g., `Jan-2016`).
- Fundamental models ignore `interest rate`, `installment`, `grade`, `sub-grade`.
- XGB (Full) uses those extra fields.
- Model output is a risk indicator and should support (not replace) lending policy decisions.

# Feature Guide (What each input means + recommended range)

## Loan & Income Information
- Loan amount (loan_amnt)
    - What it is: Total amount the borrower wants to borrow
    - Why it matters: Larger loans = higher risk (harder to repay)
    - Recommended range: $500 – $40,000

- Loan term (term)
    - What it is: Length of the loan (36 or 60 months)
    - Why it matters: Longer term → more uncertainty → higher risk
    - Options:
        -  36 months (lower risk)
        -  60 months (higher risk)

- Annual income (annual_inc)
    - What it is: Borrower’s yearly income
    - Why it matters: Higher income → better repayment ability
    - Recommended range: $10,000 – $500,000

- Employment length (emp_length)
    - What it is: Years of employment
    - Why it matters: Stable employment → lower risk
    - Recommended range: 0 – 10+ years

- Borrower Profile
    - Home ownership (home_ownership)
    - What it is: Living situation
    - Why it matters: Mortgage/own = more stable
    - Options:
        - RENT (higher risk)
        - OWN / MORTGAGE (lower risk)

- Verification status (verification_status)
    - What it is: Whether income was verified
    - Why it matters:
        - Counterintuitive — verified often means lender was cautious
    - Options:
        - Not Verified
        -  Source Verified
        -  Verified (can signal higher risk in dataset)

- Loan purpose (purpose)
    - What it is: Reason for loan
    - Why it matters: Some purposes are riskier
    - Examples:
        - debt_consolidation (common)
        - small_business (higher risk)

- Borrower state (addr_state)
    - What it is: Geographic location
    - Why it matters: Reflects economic conditions
    - Format:
        - Two-letter code (e.g., CA, NY)

## Credit Quality
- FICO range low / high (fico_range_low, fico_range_high)
    - What it is: Credit score range
    - Why it matters: One of the strongest predictors
    - Recommended range: 300 – 850
    - Interpretation:
        - <600 → high risk
        - 650–700 → moderate
        - 720 → low risk

- Earliest credit line (earliest_cr_line)
    - What it is: When borrower first got credit
    - Why it matters: Longer history → more reliable
    - Format: Mon-YYYY (e.g., Jan-2010)

- Issue date (issue_d)
    - What it is: Loan start date
    - Why it matters: Used for credit age calculation
    - Recommended: Past date (e.g., 2015–2017)

## Credit Behavior
- Open accounts (open_acc)
    - What it is: Number of active accounts
    - Why it matters: Too many → risky
    - Recommended range: 1 – 30

- Total accounts (total_acc)
    - What it is: Total credit history accounts
    - Why it matters: More history → better
    - Recommended range: 5 – 100

- Delinquencies (delinq_2yrs)
    - What it is: Late payments in last 2 years
    - Why it matters: Strong risk indicator
    - Recommended range: 0 – 5 (higher is risky)

- Public records (pub_rec)
    - What it is: Bankruptcies / legal records
    - Why it matters: Very strong negative signal
    - Recommended range: 0 – 3
    
## Debt & Utilization
- Debt-to-income (dti)
    - What it is: Debt relative to income (%)
    - Why it matters: Higher = more financial stress
    - Recommended range: 0 – 40%

- Revolving balance (revol_bal)
    - What it is: Credit card debt amount
    - Why it matters: Higher balance → riskier
    - Recommended range: $0 – $100,000

- Revolving utilization (revol_util)
    - What it is: % of credit used
    - Why it matters: One of the strongest predictors
    - Recommended range: 0% – 100%
    - Interpretation:
        - <30% → healthy
        - 30–70% → moderate
        - 80% → high risk

## Full Model ONLY (XGB Full)
- Interest rate (int_rate)
    - What it is: Loan interest rate (%)
    - Why it matters: Higher rate = riskier borrower
    - Recommended range: 5% – 30%

- Installment (installment)
    - What it is: Monthly payment
    - Why it matters: Higher payment → harder to repay
    - Recommended range: $50 – $2,000

- Grade (grade)
    - What it is: LendingClub risk grade (A–G)
    - Why it matters: Direct risk signal
    - Range: A (best) → G (worst)

- Sub-grade (sub_grade)
    - What it is: More detailed grade (A1–G5)
    - Why it matters: Fine-grained risk level
    - Format: Letter + number (e.g., C3)

# Summary: 
This dashboard demonstrates how borrower characteristics, credit history, and loan attributes influence the likelihood of loan default. By entering different inputs, users can observe how factors such as credit score, debt-to-income ratio, and repayment history affect the predicted default probability. The model outputs both a probability and a risk classification, helping users understand not only whether a loan is likely to default but also the level of risk associated with it. Overall, the dashboard highlights that loan default is driven by a combination of financial stability, credit behavior, and loan conditions, rather than any single factor.